# Part 2: How the Model Works — The Inference Story

### GenAI Masterclass — From Training to Response

---

In **Part 1**, we learned how an LLM is built:

- **Chapter 1** — The Big Picture: Two training phases (tokenizer → model)
- **Chapter 2** — Training the Tokenizer: BPE algorithm created a **vocabulary table** + **merge rules**
- **Chapter 3** — Training the Model: Next-token prediction over trillions of examples created an **embedding table** + **attention weights** + **feed-forward weights**, then SFT and RLHF turned it into a helpful assistant

Now everything is **frozen**. The model will never learn again.

So what happens when you type a message into ChatGPT or Claude? That's **inference** — and in this notebook, we'll trace the complete journey of your text through the model, step by step, with working Python code.

---

## Setup

We'll use two models in this notebook:

1. **GPT-2** (open-source, small) — to peek inside the model's internals: embedding table, attention weights, feed-forward matrices
2. **OpenAI API (GPT-4o + text-embedding-3-large)** — for production-quality similarity scores and generation

This combination lets us **see the mechanics** (GPT-2) AND **see quality results** (GPT-4).

In [ ]:
# Install required libraries (run once)
!pip install tiktoken transformers torch numpy openai matplotlib -q

In [2]:
import tiktoken
import numpy as np
import torch
import sys
import time
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from openai import OpenAI

# ── Load GPT-2 (for peeking inside the model) ──
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_model.eval()

from dotenv import load_dotenv
import os
load_dotenv(override=True)
# ── Setup OpenAI API (for production-quality results) ──
# Replace with your API key
client = OpenAI()

print("✅ GPT-2 loaded (for inspecting internals)")
print("✅ OpenAI API connected (for production-quality output)")

✅ GPT-2 loaded (for inspecting internals)
✅ OpenAI API connected (for production-quality output)


In [3]:
# ── Helper functions we'll use throughout ──

def cosine_similarity(vec_a, vec_b):
    """How similar are two vectors? 1.0 = identical, 0.0 = unrelated."""
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

def get_openai_embedding(text):
    """Get embedding from OpenAI's production embedding model (3,072 dims)."""
    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    return np.array(response.data[0].embedding)

# GPT-2 embedding table (frozen weights from training)
gpt2_embedding_table = gpt2_model.transformer.wte.weight.detach().numpy()

def get_gpt2_embedding(word):
    """Look up a word's embedding from GPT-2's frozen table (768 dims)."""
    token_id = gpt2_tokenizer.encode(word)[0]
    return gpt2_embedding_table[token_id]

print("✅ Helper functions ready")

✅ Helper functions ready


---

## Chapter 4: Text to Tokens — Tokenizer in Action

### Continuing the Story...

In Part 1 (Chapter 2), we saw how BPE training produced two things that were **frozen forever**:

1. **Vocabulary Table** — ~100K entries mapping text ↔ IDs (e.g., "hello" ↔ 15339)
2. **Merge Rules** — ~100K ordered rules (e.g., t + h → th, th + e → the)

Now, when you type a message, the **same tokenizer** uses those frozen merge rules to split your text into tokens. This is the very first thing that happens — before the model even sees your text.

Let's watch it in action.

### 4.1 — Basic Tokenization: Text → Token IDs

In [5]:
# Load the tokenizer (this is the FROZEN tokenizer from training)
enc = tiktoken.get_encoding("cl100k_base")  # Used by GPT-4, Claude uses similar

text = "Hello, hi nijanthan?"

# Step 1: Encode — apply merge rules, then look up vocabulary table
token_ids = enc.encode(text)

print(f"Input text:  '{text}'")
print(f"Token IDs:   {token_ids}")
print(f"Num tokens:  {len(token_ids)}")

Input text:  'Hello, hi nijanthan?'
Token IDs:   [9906, 11, 15960, 308, 3251, 32329, 276, 30]
Num tokens:  8


In [6]:
# Step 2: Let's see WHAT each token ID maps back to
# This is the vocabulary table lookup in reverse

print("Token-by-token breakdown:")
print("-" * 40)
for tid in token_ids:
    token_text = enc.decode([tid])
    print(f"  ID {tid:>6}  →  '{token_text}'")

Token-by-token breakdown:
----------------------------------------
  ID   9906  →  'Hello'
  ID     11  →  ','
  ID  15960  →  ' hi'
  ID    308  →  ' n'
  ID   3251  →  'ij'
  ID  32329  →  'anth'
  ID    276  →  'an'
  ID     30  →  '?'


### 4.2 — Merge Rules in Action

Remember from Part 1: BPE merge rules determine how text gets split. Common words become single tokens, rare words get split into pieces. Let's see this with different words:

In [6]:
# Common vs rare words — see how tokenization differs

test_words = [
    "the",                  # Extremely common → single token
    "hello",                # Common → single token
    "Python",               # Common in training data (lots of code)
    "transformer",          # Technical but common
    "supercalifragilistic", # Rare → split into many pieces
    "Pneumonoultramicroscopicsilicovolcanoconiosis",  # Very rare
]

print(f"{'Word':<50} {'Tokens':>6}  Token pieces")
print("=" * 90)
for word in test_words:
    ids = enc.encode(word)
    pieces = [enc.decode([tid]) for tid in ids]
    print(f"{word:<50} {len(ids):>6}  {pieces}")

Word                                               Tokens  Token pieces
the                                                     1  ['the']
hello                                                   1  ['hello']
Python                                                  1  ['Python']
transformer                                             2  ['transform', 'er']
supercalifragilistic                                    7  ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic']
Pneumonoultramicroscopicsilicovolcanoconiosis          17  ['P', 'ne', 'um', 'on', 'oul', 'tram', 'icro', 'sc', 'op', 'ics', 'il', 'ic', 'ov', 'ol', 'cano', 'con', 'iosis']


💡 **Key takeaway:** The tokenizer doesn't understand language — it's purely statistical. Words that appeared frequently in the training data have their own token IDs. Rare words get broken into subword pieces using the merge rules from BPE training.

### 4.3 — Why Tokenization Matters: Cost & Context

In [7]:
# Same meaning, very different token counts

texts = [
    "What is AI?",
    "Can you explain what artificial intelligence is?",
    "I was wondering if you could provide me with a comprehensive explanation of what artificial intelligence is and how it works in detail.",
]

print("Prompt length comparison:")
print("=" * 60)
for t in texts:
    ids = enc.encode(t)
    print(f"  {len(ids):>3} tokens | {t[:60]}{'...' if len(t)>60 else ''}")

print("\n💡 Same question, 3x–8x token cost difference!")
print("   This is why concise prompts save money.")

Prompt length comparison:
    4 tokens | What is AI?
    8 tokens | Can you explain what artificial intelligence is?
   24 tokens | I was wondering if you could provide me with a comprehensive...

💡 Same question, 3x–8x token cost difference!
   This is why concise prompts save money.


In [ ]:
# Language comparison
sentences = [
    ("English",  "The weather is nice today."),
    ("Spanish",  "El clima está agradable hoy."),
    ("Japanese", "今日はいい天気ですね。"),
    ("Arabic",   "الطقس جميل اليوم."),
    ("Code",     "print('Hello, World!')"),
]

print(f"{'Language':<12} {'Tokens':>6}  Text")
print("=" * 65)
for lang, text in sentences:
    ids = enc.encode(text)
    print(f"{lang:<12} {len(ids):>6}  {text}")

print("\n💡 Non-English text often uses 2-3x more tokens for the same meaning.")
print("   This is a direct consequence of BPE training data being English-heavy.")

---

## Chapter 5: Tokens to Meaning — Embedding Lookup

### Continuing the Story...

By the end of Phase 2 of model training (Chapter 3), the model created a final **embedding table** — a giant lookup table that maps every token ID to a **meaning vector**. This vector positions the token in a multi-dimensional space where similar words are close together.

Remember:
- During training, weights moved tokens around in this space (the WeightsDemo showed this)
- Layer after layer nudged each token's position (the Layer-by-Layer demo showed this)
- After training, the positions are **frozen**

Now at inference time, the embedding table is a simple **key-value lookup**: give it a token ID, get back a meaning vector.

### 5.1 — Peeking Inside the Embedding Table (GPT-2)

In [8]:
print(f"GPT-2 Embedding Table")
print(f"=" * 50)
print(f"Shape: {gpt2_embedding_table.shape}")
print(f"  → {gpt2_embedding_table.shape[0]:,} tokens (vocabulary size)")
print(f"  → {gpt2_embedding_table.shape[1]} dimensions per token")
print(f"  → Total numbers stored: {gpt2_embedding_table.shape[0] * gpt2_embedding_table.shape[1]:,}")
print(f"\nThis is a 2D table: each row is one token, each column is one dimension.")
print(f"To get a token's meaning, you just fetch its row — a simple lookup.")

GPT-2 Embedding Table
Shape: (50257, 768)
  → 50,257 tokens (vocabulary size)
  → 768 dimensions per token
  → Total numbers stored: 38,597,376

This is a 2D table: each row is one token, each column is one dimension.
To get a token's meaning, you just fetch its row — a simple lookup.


### 5.2 — Looking Up a Token's Meaning Vector

In [9]:
word = "king"
token_id = gpt2_tokenizer.encode(word)[0]
embedding_vector = gpt2_embedding_table[token_id]

print(f"Word: '{word}'")
print(f"Token ID: {token_id}")
print(f"Embedding vector (first 20 of {len(embedding_vector)} dimensions):")
print(f"  {embedding_vector[:20].round(4)}")
print(f"\n💡 This vector IS the meaning of '{word}' — frozen in place during training.")
print(f"   Every time '{word}' appears, it starts with this exact same vector.")
print(f"   Then transformer layers (attention + feed-forward) refine it for context.")

Word: 'king'
Token ID: 3364
Embedding vector (first 20 of 768 dimensions):
  [ 0.1093 -0.3947  0.0594 -0.1016 -0.2786  0.2597 -0.3124 -0.0881  0.0112
 -0.1193  0.0201  0.0891  0.1486  0.1545  0.2899  0.1375 -0.0431  0.132
 -0.0053  0.0159]

💡 This vector IS the meaning of 'king' — frozen in place during training.
   Every time 'king' appears, it starts with this exact same vector.
   Then transformer layers (attention + feed-forward) refine it for context.


### 5.3 — Cosine Similarity: Model Size Matters

In Part 1, we said training positions related words close together. Let's verify — first with GPT-2's embeddings (768 dimensions), then with a modern production model (3,072 dimensions).

**Why show both?** Because model size matters enormously. GPT-2 had to squeeze 50,257 tokens into 768 dimensions — like seating 50,000 people in a school gym. A larger model has more room — like a stadium.

In [10]:
# ── PART A: GPT-2 Embeddings (768 dimensions, 2019, 124M params) ──

pairs = [
    ("king",  "queen",    "Related — royalty"),
    ("king",  "prince",   "Related — royalty"),
    ("dog",   "puppy",    "Related — animals"),
    ("cat",   "kitten",   "Related — animals"),
    ("Paris", "France",   "Related — geography"),
    ("happy", "sad",      "Related — emotions (opposites)"),
    ("king",  "banana",   "Unrelated"),
    ("dog",   "computer", "Unrelated"),
    ("Paris", "puppy",    "Unrelated"),
]

print("📊 GPT-2 Embeddings (768 dimensions)")
print("   Model: GPT-2 Small | Released: 2019 | Parameters: 124M")
print("=" * 65)
print(f"{'Word A':<10} {'Word B':<10} {'Similarity':>10}  Relationship")
print("-" * 65)

gpt2_scores = {}
for w1, w2, label in pairs:
    sim = cosine_similarity(get_gpt2_embedding(w1), get_gpt2_embedding(w2))
    gpt2_scores[(w1,w2)] = sim
    bar = "█" * int(abs(sim) * 20)
    print(f"{w1:<10} {w2:<10} {sim:>10.4f}  {bar} {label}")

print("\n🤔 Notice: king–queen (0.27) is LOWER than dog–computer (0.33).")
print("   The embeddings ARE trained (not random), but GPT-2 is tiny.")
print("   768 dimensions for 50K tokens = very crowded space.")
print("\n   Let's see what a modern, larger model looks like... ↓")

📊 GPT-2 Embeddings (768 dimensions)
   Model: GPT-2 Small | Released: 2019 | Parameters: 124M
Word A     Word B     Similarity  Relationship
-----------------------------------------------------------------
king       queen          0.2666  █████ Related — royalty
king       prince         0.2836  █████ Related — royalty
dog        puppy          0.2618  █████ Related — animals
cat        kitten         0.2705  █████ Related — animals
Paris      France         0.6505  █████████████ Related — geography
happy      sad            0.2160  ████ Related — emotions (opposites)
king       banana         0.2280  ████ Unrelated
dog        computer       0.3294  ██████ Unrelated
Paris      puppy          0.2233  ████ Unrelated

🤔 Notice: king–queen (0.27) is LOWER than dog–computer (0.33).
   The embeddings ARE trained (not random), but GPT-2 is tiny.
   768 dimensions for 50K tokens = very crowded space.

   Let's see what a modern, larger model looks like... ↓


In [11]:
# ── PART B: OpenAI text-embedding-3-large (3,072 dimensions) ──

print("Fetching embeddings from OpenAI text-embedding-3-large...")
all_words = list(set(w for p in pairs for w in [p[0], p[1]]))
openai_embeddings = {}
for word in all_words:
    openai_embeddings[word] = get_openai_embedding(word)

emb_dim = len(openai_embeddings[all_words[0]])

print(f"\n📊 OpenAI text-embedding-3-large ({emb_dim:,} dimensions)")
print(f"   4× more dimensions = 4× more room to organize tokens")
print("=" * 65)
print(f"{'Word A':<10} {'Word B':<10} {'Similarity':>10}  Relationship")
print("-" * 65)

oai_scores = {}
for w1, w2, label in pairs:
    sim = cosine_similarity(openai_embeddings[w1], openai_embeddings[w2])
    oai_scores[(w1,w2)] = sim
    bar = "█" * int(abs(sim) * 20)
    print(f"{w1:<10} {w2:<10} {sim:>10.4f}  {bar} {label}")

print("\n✅ NOW the scores match our intuition!")
print("   Related words → high similarity. Unrelated words → low similarity.")

Fetching embeddings from OpenAI text-embedding-3-large...

📊 OpenAI text-embedding-3-large (3,072 dimensions)
   4× more dimensions = 4× more room to organize tokens
Word A     Word B     Similarity  Relationship
-----------------------------------------------------------------
king       queen          0.5525  ███████████ Related — royalty
king       prince         0.5030  ██████████ Related — royalty
dog        puppy          0.5768  ███████████ Related — animals
cat        kitten         0.5646  ███████████ Related — animals
Paris      France         0.6298  ████████████ Related — geography
happy      sad            0.5286  ██████████ Related — emotions (opposites)
king       banana         0.3272  ██████ Unrelated
dog        computer       0.3892  ███████ Unrelated
Paris      puppy          0.1856  ███ Unrelated

✅ NOW the scores match our intuition!
   Related words → high similarity. Unrelated words → low similarity.


In [12]:
# ── PART C: Head-to-Head Comparison ──

print("📊 HEAD-TO-HEAD: GPT-2 (768d) vs OpenAI (3,072d)")
print("=" * 82)
print(f"{'Word A':<10} {'Word B':<10} {'GPT-2':>8} {'OpenAI':>8}  {'Winner':>8}  Relationship")
print("-" * 82)

for w1, w2, label in pairs:
    g = gpt2_scores[(w1,w2)]
    o = oai_scores[(w1,w2)]
    # Better = higher for related, lower for unrelated
    if "Unrelated" in label:
        better = "OpenAI ✅" if o < g else "GPT-2"
    else:
        better = "OpenAI ✅" if o > g else "GPT-2"
    print(f"{w1:<10} {w2:<10} {g:>8.4f} {o:>8.4f}  {better:>10}  {label}")

print()
print("💡 KEY TAKEAWAY:")
print("   The same training principle (next-token prediction → position related words closer)")
print("   produces dramatically better results with larger models.")
print(f"   • GPT-2:  768 dims, 50K tokens → crowded school gym")
print(f"   • OpenAI: {emb_dim:,} dims, 100K+ tokens → spacious stadium")
print(f"   More dimensions = more room = cleaner separation of meanings.")
print(f"\n   From this point forward, we'll use the production models for cleaner results.")

📊 HEAD-TO-HEAD: GPT-2 (768d) vs OpenAI (3,072d)
Word A     Word B        GPT-2   OpenAI    Winner  Relationship
----------------------------------------------------------------------------------
king       queen        0.2666   0.5525    OpenAI ✅  Related — royalty
king       prince       0.2836   0.5030    OpenAI ✅  Related — royalty
dog        puppy        0.2618   0.5768    OpenAI ✅  Related — animals
cat        kitten       0.2705   0.5646    OpenAI ✅  Related — animals
Paris      France       0.6505   0.6298       GPT-2  Related — geography
happy      sad          0.2160   0.5286    OpenAI ✅  Related — emotions (opposites)
king       banana       0.2280   0.3272       GPT-2  Unrelated
dog        computer     0.3294   0.3892       GPT-2  Unrelated
Paris      puppy        0.2233   0.1856    OpenAI ✅  Unrelated

💡 KEY TAKEAWAY:
   The same training principle (next-token prediction → position related words closer)
   produces dramatically better results with larger models.
   • GPT-2:

### 5.4 — Vector Arithmetic: king - man + woman = ?

The most famous embedding demonstration: relationships are encoded as **directions** in the vector space. If training worked properly, the direction from "man" to "king" (royalty) should be the same direction as "woman" to "queen".

In [13]:
king_vec = get_openai_embedding("king")
man_vec = get_openai_embedding("man")
woman_vec = get_openai_embedding("woman")

result_vector = king_vec - man_vec + woman_vec

candidates = ["queen", "princess", "duchess", "prince", "empress",
              "dog", "banana", "computer", "house", "river"]

print("king - man + woman = ?")
print("=" * 55)

results = []
for word in candidates:
    word_vec = get_openai_embedding(word)
    sim = cosine_similarity(result_vector, word_vec)
    results.append((word, sim))

results.sort(key=lambda x: x[1], reverse=True)

for i, (word, sim) in enumerate(results):
    bar = "█" * int(max(sim, 0) * 30)
    marker = " ← 👑" if i == 0 else ""
    print(f"  {i+1:>2}. {word:<12} {sim:.4f}  {bar}{marker}")

print()
print("💡 The direction from 'man' → 'king' encodes the concept of 'royalty'.")
print("   Applying that same direction to 'woman' gives us 'queen'.")
print("   This is proof that training organized the embedding space meaningfully.")

king - man + woman = ?
   1. queen        0.5213  ███████████████ ← 👑
   2. empress      0.4248  ████████████
   3. princess     0.4211  ████████████
   4. duchess      0.3526  ██████████
   5. prince       0.3433  ██████████
   6. river        0.2463  ███████
   7. house        0.2296  ██████
   8. banana       0.2166  ██████
   9. dog          0.2157  ██████
  10. computer     0.2005  ██████

💡 The direction from 'man' → 'king' encodes the concept of 'royalty'.
   Applying that same direction to 'woman' gives us 'queen'.
   This is proof that training organized the embedding space meaningfully.


### 5.5 — Important Limitation: No Context Yet!

The embedding table gives each token its **general** meaning. But "bank" means something different in "river bank" vs "bank account". 

Example : **"I sat by the river bank"**

At this stage, the model doesn't know bank relates to river — that disambiguation comes from self-attention in the next chapter.

In [1]:
bank_vec = get_openai_embedding("bank")

context_words = ["river", "water", "shore", "money", "account", "financial", "loan"]

print("Similarity of 'bank' to various words:")
print("(Static embedding — no context applied yet!)")
print("-" * 50)
for w in context_words:
    sim = cosine_similarity(bank_vec, get_openai_embedding(w))
    bar = "█" * int(max(sim, 0) * 30)
    print(f"  bank ↔ {w:<12} {sim:.4f}  {bar}")

print()
print("⚠️  Without context, 'bank' is similar to BOTH financial and river meanings.")
print("   Self-attention (Chapter 6) will disambiguate based on surrounding words.")

NameError: name 'get_openai_embedding' is not defined